# NovaPerakende · 2024 satış ve nüfus analizi

İl bazında toplam net satış ve 1.000 kişi başına net satışı karşılaştırır.

In [ ]:
from pathlib import Path
from collections import defaultdict
from decimal import Decimal
import csv

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dbt").is_dir())

def load(name):
    with (root / "data" / "processed" / name).open(encoding="utf-8-sig", newline="") as file:
        return [row for row in csv.DictReader(file) if row["yil"] == "2024"]

cities = load("mart_sehir_performans.csv")
population = {row["sehir"]: int(row["nufus"]) for row in cities}
assert cities and len(population) == len(cities) and min(population.values()) > 0
sales = defaultdict(Decimal)
for row in load("fct_satis.csv"):
    sales[row["sehir"]] += Decimal(row["net_satis"])
assert sales.keys() <= population.keys(), "Nüfus eşleşmesi eksik"
assert all(sales.get(row["sehir"], 0) == Decimal(row["net_satis"]) for row in cities)
print(f"İl sayısı: {len(cities)} | Toplam net satış: {sum(sales.values()):,.2f} TL")

In [ ]:
for metric, label in [("net_satis", "Toplam net satış"),
                      ("bin_kisiye_satis", "1.000 kişi başına net satış")]:
    print(f"\n{label}")
    top = sorted(cities, key=lambda row: (-Decimal(row[metric]), row["sehir"]))[:10]
    peak = Decimal(top[0][metric])
    for row in top:
        value = Decimal(row[metric])
        bar = "█" * int(value / peak * 25) if peak > 0 else ""
        print(f'{row["sehir"]:18} {value:>15,.2f} TL  {bar}')

Toplam satış hacmi ve nüfusa göre satış yoğunluğu farklı ölçütlerdir. Sıralama farklarını yorumlamak için mağaza erişimi ve kanal dağılımı ayrıca incelenebilir.